# Pertemuan 14
## Transportation in Supply Chains

Optimasi rute distribusi beras dari sentra produksi ke Jakarta.

---
## 1. Setup

In [ ]:
# %pip install ortools

   ---------------------------------------- 0.0/24.7 MB ? eta -:--:--
   -- ------------------------------------- 1.8/24.7 MB 14.1 MB/s eta 0:00:02
   ------- -------------------------------- 4.5/24.7 MB 13.3 MB/s eta 0:00:02
   ----------- ---------------------------- 7.1/24.7 MB 13.2 MB/s eta 0:00:02
   ---------------- ----------------------- 10.0/24.7 MB 13.1 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.7 MB 13.1 MB/s eta 0:00:01
   ------------------------ --------------- 15.2/24.7 MB 13.1 MB/s eta 0:00:01
   ---------------------------- ----------- 17.8/24.7 MB 13.1 MB/s eta 0:00:01
   --------------------------------- ------ 20.4/24.7 MB 13.0 MB/s eta 0:00:01
   ------------------------------------- -- 23.1/24.7 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 24.7/24.7 MB 12.8 MB/s  0:00:01

   ---------------------------------------- 0/4 [protobuf]
   ---------------------------------------- 0/4 [protobuf]
   ------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 2. Dataset Rute

In [1]:
import pandas as pd

rute = pd.DataFrame({
    'asal':          ['Karawang', 'Indramayu', 'Cirebon', 'Lampung'],
    'tujuan':        ['Jakarta'] * 4,
    'jarak_km':      [80,         200,         250,        350],
    'biaya_rp_ton':  [500000,     900000,      1100000,    1800000],
    'waktu_jam':     [2,          5,           6,          10],
    'kapasitas_ton': [50,         80,          80,         120],
    'stok_tersedia': [200,        350,         300,        500],   # ton
})

rute

,asal,tujuan,jarak_km,biaya_rp_ton,waktu_jam,kapasitas_ton,stok_tersedia
0,Karawang,Jakarta,80,500000,2,50,200
1,Indramayu,Jakarta,200,900000,5,80,350
2,Cirebon,Jakarta,250,1100000,6,80,300
3,Lampung,Jakarta,350,1800000,10,120,500


---
## 3. Optimasi Rute dengan OR-Tools

Target: penuhi kebutuhan Jakarta 600 ton dengan biaya minimum dan waktu ≤ 1 hari (24 jam).

In [2]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')

KEBUTUHAN = 600  # ton
BATAS_WAKTU = 24  # jam

# Variabel: jumlah ton yang dikirim dari setiap asal
x = [
    solver.NumVar(0.0, float(rute.loc[i, 'stok_tersedia']), f"x_{rute.loc[i, 'asal']}")
    for i in range(len(rute))
]

# Constraint 1: total pengiriman >= kebutuhan
solver.Add(sum(x) >= KEBUTUHAN)

# Constraint 2: waktu pengiriman <= batas waktu
for i in range(len(rute)):
    solver.Add(x[i] * rute.loc[i, 'waktu_jam'] / rute.loc[i, 'kapasitas_ton'] <= BATAS_WAKTU)

# Objective: minimasi biaya
solver.Minimize(sum(x[i] * rute.loc[i, 'biaya_rp_ton'] for i in range(len(rute))))

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solusi optimal ditemukan!")
    print(f"Total biaya: Rp {solver.Objective().Value():,.0f}\n")
    hasil = []
    for i in range(len(rute)):
        if x[i].solution_value() > 0:
            hasil.append({
                'asal': rute.loc[i, 'asal'],
                'jumlah_ton': round(x[i].solution_value(), 1),
                'biaya_rp': round(x[i].solution_value() * rute.loc[i, 'biaya_rp_ton']),
                'waktu_jam': rute.loc[i, 'waktu_jam'],
            })
    print(pd.DataFrame(hasil))
else:
    print("Solusi tidak ditemukan.")

Solusi optimal ditemukan!
Total biaya: Rp 470,000,000

        asal  jumlah_ton   biaya_rp  waktu_jam
0   Karawang       200.0  100000000          2
1  Indramayu       350.0  315000000          5
2    Cirebon        50.0   55000000          6


---
## 4. AI Agent: Rekomendasi Rute (Groq)

In [3]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

data_str = rute.to_string(index=False)

prompt = f"""
Data rute distribusi beras ke Jakarta:
{data_str}

Kebutuhan Jakarta: 600 ton
Batas waktu pengiriman: 1 hari

Cari rute distribusi paling murah dengan batas waktu pengiriman 1 hari.
Berikan:
1. Rute yang dipilih dan alasannya
2. Total biaya estimasi
3. Rekomendasi untuk efisiensi jangka panjang
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Berikut adalah jawaban untuk pertanyaan Anda:

**1. Rute yang dipilih dan alasannya:**

Untuk memenuhi kebutuhan 600 ton dengan batas waktu pengiriman 1 hari, kita perlu memilih rute yang paling murah dan efektif. Berdasarkan data, kita dapat melihat bahwa rute Karawang-Jakarta memiliki biaya terendah per ton (Rp 500.000) dan waktu pengiriman tercepat (2 jam).

Namun, kapasitas rute Karawang-Jakarta hanya 50 ton, sehingga kita perlu memilih rute lain untuk memenuhi kebutuhan yang tersisa. Rute Indramayu-Jakarta memiliki kapasitas yang lebih besar (80 ton) dan biaya yang lebih rendah dibandingkan dengan rute Cirebon-Jakarta dan Lampung-Jakarta.

Dengan demikian, rute yang dipilih adalah:
- Karawang-Jakarta (200 ton, karena stok tersedia 200 ton)
- Indramayu-Jakarta (400 ton, karena kebutuhan yang tersisa 600 - 200 = 400 ton dan kapasitas rute Karawang sudah mencapai stok tersedia)

Alasannya adalah bahwa rute ini memiliki biaya yang relatif lebih rendah dan dapat memenuhi kebutuhan 600 